In [0]:
%pip install databricks-agents mlflow pyspark
dbutils.library.restartPython()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType, StringType, DateType, TimestampType

def compute_report(table_name):
    """Compute data quality report for any table"""
    try:
        df = spark.sql(f"SELECT * FROM {table_name}")

        total_rows      = df.count()
        total_cols      = len(df.columns)
        duplicate_count = total_rows - df.dropDuplicates().count()

        # NULL checks
        null_exprs  = [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
        null_counts = df.select(null_exprs).collect()[0].asDict()
        null_issues = [
            f"  {col}: {cnt} NULLs ({round(cnt/total_rows*100,1)}%)"
            for col, cnt in null_counts.items() if cnt > 0
        ]

        # Numeric stats
        numeric_cols    = [f.name for f in df.schema.fields if isinstance(f.dataType, NumericType)]
        numeric_stats   = []
        negative_issues = []
        if numeric_cols:
            stats_df = df.select(numeric_cols).summary("min", "max", "mean")
            stats    = {row["summary"]: row.asDict() for row in stats_df.collect()}
            for col in numeric_cols:
                mn  = round(float(stats["min"][col]),  2) if stats["min"][col]  else "N/A"
                mx  = round(float(stats["max"][col]),  2) if stats["max"][col]  else "N/A"
                avg = round(float(stats["mean"][col]), 2) if stats["mean"][col] else "N/A"
                numeric_stats.append(f"  {col} -> min: {mn}, max: {mx}, avg: {avg}")
                neg = df.filter(F.col(col) < 0).count()
                if neg > 0:
                    negative_issues.append(f"  {col}: {neg} negative values found")

        # Date ranges
        date_cols  = [f.name for f in df.schema.fields if isinstance(f.dataType, (DateType, TimestampType))]
        date_stats = []
        for col in date_cols:
            mn = df.select(F.min(col)).collect()[0][0]
            mx = df.select(F.max(col)).collect()[0][0]
            date_stats.append(f"  {col} -> from: {mn}  to: {mx}")

        # String uniqueness
        string_cols  = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
        string_stats = []
        for col in string_cols[:5]:
            unique = df.select(col).distinct().count()
            string_stats.append(f"  {col} -> {unique} unique values")

        # Health score
        total_nulls  = sum(null_counts.values())
        total_cells  = total_rows * total_cols
        null_pct     = round(total_nulls / total_cells * 100, 2) if total_cells > 0 else 0
        health_score = round(100 - null_pct - (duplicate_count / total_rows * 10), 1)
        health_score = max(0, min(100, health_score))
        health_label = "GOOD" if health_score >= 80 else "AVERAGE" if health_score >= 50 else "POOR"

        short_name = table_name.split(".")[-1]
        lines = [
            f"Data Quality Report - {short_name}",
            "=" * 45,
            "",
            "Overview",
            f"  Total Rows    : {total_rows:,}",
            f"  Total Columns : {total_cols}",
            f"  Duplicates    : {duplicate_count if duplicate_count > 0 else 'None'}",
            "",
            "NULL Analysis",
        ]
        lines += null_issues if null_issues else ["  No NULLs found"]
        if numeric_stats:
            lines += ["", "Numeric Column Stats"] + numeric_stats
        if negative_issues:
            lines += ["", "Negative Value Issues"] + negative_issues
        if date_stats:
            lines += ["", "Date Column Ranges"] + date_stats
        if string_stats:
            lines += ["", "String Column Uniqueness"] + string_stats
        lines += [
            "",
            f"Overall Health Score : {health_score}/100 ({health_label})",
            "=" * 45
        ]
        return "\n".join(lines)

    except Exception as e:
        return f"Error computing report for {table_name}: {str(e)}"


#  Pre-compute reports for all tables
CATALOG  = "data_engineering_workshop"
SCHEMA   = "custom_agent_schema"

# List all tables in schema
tables_df = spark.sql(f"SHOW TABLES IN {CATALOG}.{SCHEMA}")
all_tables = [f"{CATALOG}.{SCHEMA}.{row.tableName}" for row in tables_df.collect()]

print(f"Found {len(all_tables)} tables: {all_tables}")

# Pre-compute report for each table
REPORTS = {}
for table in all_tables:
    short_name = table.split(".")[-1]
    print(f"Computing report for {short_name}...")
    REPORTS[short_name] = compute_report(table)

print(f"Available tables: {list(REPORTS.keys())}")

In [0]:
from mlflow.pyfunc import PythonModel
import pandas as pd
import json
import mlflow

class DataQualityAgent(PythonModel):
    def __init__(self, reports):
        self.reports = reports  # dict of {table_name: report_string}

    def predict(self, context, model_input):
        try:
            if isinstance(model_input, pd.DataFrame):
                messages = model_input["messages"].iloc[0]
                if isinstance(messages, str):
                    messages = json.loads(messages)
                user_query = messages[-1]["content"].lower()
            elif isinstance(model_input, dict):
                user_query = model_input["messages"][-1]["content"].lower()
            else:
                user_query = ""
        except:
            user_query = ""

        # Detect table name from prompt
        matched_table = None
        for table_name in self.reports.keys():
            if table_name.lower() in user_query:
                matched_table = table_name
                break

        #  List tables prompt
        if any(word in user_query for word in ["list tables", "show tables", "available tables", "which tables"]):
            table_list = "\n".join([f"  - {t}" for t in self.reports.keys()])
            content = f"Available tables:\n{table_list}\n\nTry: 'check ec_orders quality'"

        #  Table-specific quality check
        elif matched_table and any(word in user_query for word in ["quality", "check", "report", "metrics", "analyze"]):
            content = self.reports[matched_table]

        #  Default table (ec_orders) quality check
        elif any(word in user_query for word in ["quality", "check", "report", "metrics", "analyze"]):
            default = list(self.reports.keys())[0]
            content = f"No table specified. Showing report for '{default}':\n\n" + self.reports[default]

        else:
            table_list = "\n".join([f"  - check {t} quality" for t in self.reports.keys()])
            content = (
                "I am your Data Quality Agent.\n\n"
                "Try asking:\n"
                f"{table_list}\n"
                "  - list tables\n"
                "  - check data quality"
            )

        return {
            "choices": [
                {
                    "message": {
                        "role": "assistant",
                        "content": content
                    },
                    "finish_reason": "stop",
                    "index": 0
                }
            ]
        }

mlflow.models.set_model(DataQualityAgent(REPORTS))
print("Agent ready with tables:", list(REPORTS.keys()))
print("\n\n")

In [0]:
import mlflow

mlflow.set_registry_uri("databricks-uc")

input_example = {
    "messages": [{"role": "user", "content": "check ec_orders quality"}]
}

signature = mlflow.models.infer_signature(
    model_input=input_example,
    model_output="Data Quality Report"
)

with mlflow.start_run():
    mlflow.pyfunc.log_model(
        name="agent",
        python_model=DataQualityAgent(REPORTS),
        registered_model_name="data_engineering_workshop.custom_agent_schema.data_quality_agent",
        input_example=input_example,
        signature=signature
    )

In [0]:
import mlflow
from databricks import agents

client = mlflow.MlflowClient()
versions = client.search_model_versions(
    "name='data_engineering_workshop.custom_agent_schema.data_quality_agent'"
)
latest_version = max([int(v.version) for v in versions])
print(f"Deploying version: {latest_version}")

agents.deploy(
    model_name="data_engineering_workshop.custom_agent_schema.data_quality_agent",
    model_version=str(latest_version)
)
